# Group 4 Exploratory Visualization: Does weather raise self driving crash risk?

DATASCI 209, Fall 2026. Hypothesis owner: Solomon Azariah. Team: Solomon Azariah, Sam Ho Der, Rich J Shelby, Paul Terrasi.

**Hypothesis.** Self driving vehicles crash more often in rain than their exposure to rain would predict: the share of reported ADS crashes in rain is higher than the share of time it rains where they operate.

**Data.** NHTSA Standing General Order ADS incident reports (archive Jul 2021 to Jun 2025 and the current amended order file, Apr 2025 onward), merged to one row per incident (2,945 crashes), plus a monthly rain exposure table for six metros built from the Open-Meteo historical archive.

**Result.** Rejected, and the opposite holds: in every metro the share of crashes in rain is about half the share of time it rains, the pattern holds month by month, rain crashes are no more severe, and no operator or season breaks it. The likely explanation is exposure (fleets drive fewer miles in rain), which the reports cannot confirm because miles by weather are not reported.

Run order: install, clean and merge, then the seven exploration steps. Charts save to `charts/`. The written report is `Group4_Exploratory_Visualization_Weather.docx`.

In [ ]:
!pip -q install altair vl-convert-python
import os; os.makedirs('charts', exist_ok=True)

## 0. Upload the data

Run the next cell and pick the files when the chooser appears. You can do this one of two ways:

- **Quick (recommended):** upload `ads_clean.csv` and `rain_exposure_monthly.csv`. The cleaning step is then skipped and the charts run from the cleaned table.
- **Full reproduction:** upload the two raw NHTSA files (`SGO-2021-01_Incident_Reports_ADS_archive_2021_2025.csv` and `SGO-2021-01_Incident_Reports_ADS_current.csv`) plus `rain_exposure_monthly.csv`. The cleaning step then rebuilds `ads_clean.csv` from scratch.

All of these are in the team repo under `weather_eda/` (and `weather_eda/data/`).

In [ ]:
import os, shutil
from google.colab import files

os.makedirs("data", exist_ok=True)
print("Choose ads_clean.csv and rain_exposure_monthly.csv (or the two raw NHTSA CSVs plus rain_exposure_monthly.csv).")
uploaded = files.upload()

for name in uploaded:
    if name.startswith("SGO-2021-01") or name == "rain_exposure_monthly.csv":
        shutil.move(name, os.path.join("data", name))

have_clean = os.path.exists("ads_clean.csv")
have_raw = all(os.path.exists(f"data/{f}") for f in
               ["SGO-2021-01_Incident_Reports_ADS_archive_2021_2025.csv", "SGO-2021-01_Incident_Reports_ADS_current.csv"])
have_rain = os.path.exists("data/rain_exposure_monthly.csv")

print("\nFound:")
print("  ads_clean.csv              ", "yes" if have_clean else "no")
print("  raw NHTSA files in data/   ", "yes" if have_raw else "no")
print("  rain_exposure_monthly.csv  ", "yes" if have_rain else "no (required)")
if not have_rain:
    raise SystemExit("rain_exposure_monthly.csv is required for the charts. Run this cell again and upload it.")
if not (have_clean or have_raw):
    raise SystemExit("Upload either ads_clean.csv or the two raw NHTSA files.")
print("\nReady." + ("" if have_raw else " The cleaning cell will be skipped because ads_clean.csv was uploaded."))

## 1. Clean and merge the two NHTSA files
Runs only if the raw NHTSA files were uploaded; otherwise the uploaded `ads_clean.csv` is used as is. Keeps the latest version of each report and one row per incident; harmonizes weather, surface, lighting and severity across the two schemas; derives month, hour, operator, metro and crash partner.

In [ ]:
import os
if os.path.exists("ads_clean.csv") and not all(os.path.exists(f"data/{f}") for f in
        ["SGO-2021-01_Incident_Reports_ADS_archive_2021_2025.csv", "SGO-2021-01_Incident_Reports_ADS_current.csv"]):
    print("ads_clean.csv already uploaded, skipping the cleaning step.")
else:
    exec(r'''
import re, pandas as pd, numpy as np
ARCHIVE = "data/SGO-2021-01_Incident_Reports_ADS_archive_2021_2025.csv"
CURRENT = "data/SGO-2021-01_Incident_Reports_ADS_current.csv"
OUT = "ads_clean.csv"

A = pd.read_csv(ARCHIVE, low_memory=False, encoding="utf-8-sig")
C = pd.read_csv(CURRENT, low_memory=False, encoding="utf-8-sig")
A["source_file"] = "archive_2021_2025"
C["source_file"] = "current_2025_on"

# 1. keep the latest version of each report, then one row per real-world incident
def latest(df):
    df = df.sort_values(["Report ID", "Report Version"]).drop_duplicates("Report ID", keep="last")
    return df.drop_duplicates("Same Incident ID", keep="first")
A, C = latest(A), latest(C)

# 2. harmonize column names between schemas
def yes(df, col):
    return df[col].astype(str).str.strip().str.upper().eq("Y") if col in df else pd.Series(False, index=df.index)

def primary_weather(df, flags):
    """flags: list of (column, label) in priority order; worst condition wins when several are ticked."""
    out = pd.Series("Unknown", index=df.index, dtype=object)
    for col, label in reversed(flags):   # iterate low priority first so high priority overwrites
        out = out.where(~yes(df, col), label)
    return out

A_flags = [("Weather - Snow", "Snow"), ("Weather - Rain", "Rain"), ("Weather - Fog/Smoke", "Fog or smoke"),
           ("Weather - Severe Wind", "Severe wind"), ("Weather - Cloudy", "Cloudy"), ("Weather - Clear", "Clear"),
           ("Weather - Other", "Other"), ("Weather - Unknown", "Unknown")]
C_flags = [("Weather - Snow", "Snow"), ("Weather - Rain", "Rain"), ("Weather - Fog/Smoke/Haze", "Fog or smoke"),
           ("Weather - Severe Wind", "Severe wind"), ("Weather - Dust Storm", "Other"), ("Weather - Severe Hurricane", "Other"),
           ("Weather - Structure-Indoor", "Other"), ("Weather - Cloudy", "Cloudy"), ("Weather - Partly Cloudy", "Cloudy"),
           ("Weather - Clear", "Clear"), ("Weather - Unk - See Narrative", "Unknown")]

A["weather"] = primary_weather(A, A_flags)
C["weather"] = primary_weather(C, C_flags)
# archive "Other" with text "Thundering" is rain weather in practice
A.loc[A["Weather - Other Text"].astype(str).str.contains("Thunder", case=False, na=False), "weather"] = "Rain"

A["surface_wet"] = A["Roadway Surface"].isin(["Wet", "Snow / Slush / Ice"])
C["surface_wet"] = yes(C, "Roadway-Wet Surface Condition")
A["surface"] = A["Roadway Surface"].fillna("Unknown").replace({"Snow / Slush / Ice": "Snow or ice", "Other, see Narrative": "Other"})
C["surface"] = np.where(C["surface_wet"], "Wet", np.where(yes(C, "Roadway-Unknown"), "Unknown", "Dry"))

A["lighting"] = A["Lighting"].fillna("Unknown").replace({"Dark - Unknown Lighting": "Dark - Lighted", "Other, see Narrative": "Unknown"})
C["lighting"] = np.nan   # not collected under the amended order

sev_map = {
    "No Injuries Reported": "No injury", "Property Damage. No Injured Reported": "No injury", "No Injured Reported": "No injury",
    "Minor": "Minor", "Minor W/O Hospitalization": "Minor", "Minor W/ Hospitalization": "Minor",
    "Moderate": "Moderate", "Moderate W/ Hospitalization": "Moderate", "Moderate W/O Hospitalization": "Moderate",
    "Serious": "Serious", "Serious W/ Hospitalization": "Serious", "Fatality": "Fatality", "Unknown": "Unknown",
}
for df in (A, C):
    df["severity"] = df["Highest Injury Severity Alleged"].map(sev_map).fillna("Unknown")
    df["any_injury"] = df["severity"].isin(["Minor", "Moderate", "Serious", "Fatality"])
    df["hospital_or_worse"] = df["Highest Injury Severity Alleged"].astype(str).str.contains("W/ Hospitalization|Serious|Fatality|Moderate", regex=True)

keep = ["Report ID", "Same Incident ID", "Reporting Entity", "Operating Entity", "Make", "Model", "Model Year",
        "Incident Date", "Incident Time (24:00)", "City", "State", "Roadway Type", "Crash With",
        "Highest Injury Severity Alleged", "SV Pre-Crash Movement", "CP Pre-Crash Movement", "SV Precrash Speed (MPH)",
        "Within ODD?", "Narrative", "weather", "surface", "surface_wet", "lighting", "severity", "any_injury",
        "hospital_or_worse", "source_file"]
A2 = A.reindex(columns=keep + ["Posted Speed Limit (MPH)"])
C2 = C.reindex(columns=keep + ["Posted Speed Limit (MPH)"])
df = pd.concat([A2, C2], ignore_index=True)

# 3. derived fields
df["month"] = pd.to_datetime(df["Incident Date"], format="%b-%Y", errors="coerce")
df["year"] = df["month"].dt.year
t = pd.to_datetime(df["Incident Time (24:00)"], format="%H:%M", errors="coerce")
df["hour"] = t.dt.hour
df["daypart"] = pd.cut(df["hour"], bins=[-1, 5, 9, 15, 19, 23], labels=["Night (0-5)", "Morning (6-9)", "Midday (10-15)", "Evening (16-19)", "Night (20-23)"])
df["adverse_weather"] = df["weather"].isin(["Rain", "Snow", "Fog or smoke", "Severe wind"])
df["operator"] = df["Reporting Entity"].str.replace(r",? (LLC|Inc\.?|Operations|Motor Company)$", "", regex=True).str.strip()
df["operator"] = df["operator"].replace({"General Motors": "Cruise (GM)", "Cruise": "Cruise (GM)", "Transdev Alternative Services": "Transdev"})
df["city"] = df["City"].str.strip().str.title()
city_fix = {"Sf": "San Francisco", "San Fransisco": "San Francisco", "Mountain View": "Mountain View"}
df["city"] = df["city"].replace(city_fix)
df["crash_partner"] = df["Crash With"].fillna("Unknown").replace({
    "Non-Motorist: Cyclist": "Cyclist", "Non-Motorist: Pedestrian": "Pedestrian", "Non-Motorist: Other": "Other non-motorist",
    "Other, see Narrative": "Other", "Other Fixed Object": "Fixed object", "Pole / Tree": "Fixed object"})
df["speed_mph"] = pd.to_numeric(df["SV Precrash Speed (MPH)"], errors="coerce")

df.to_csv(OUT, index=False)
print("archive incidents", len(A2), "current incidents", len(C2), "total", len(df))
print("weather", df["weather"].value_counts().to_dict())
print("surface", df["surface"].value_counts().to_dict())
print("severity", df["severity"].value_counts().to_dict())
print("cities", df["city"].value_counts().head(8).to_dict())
print("operators", df["operator"].value_counts().head(8).to_dict())
print("hour missing", df["hour"].isna().sum(), "month range", df["month"].min().date(), df["month"].max().date())

''')

## 2. Setup for the charts

In [ ]:
import pandas as pd, numpy as np, altair as alt, os, json
from math import sqrt
os.makedirs("charts", exist_ok=True)
alt.data_transformers.disable_max_rows()

df = pd.read_csv("ads_clean.csv", parse_dates=["month"])
ex = pd.read_csv("data/rain_exposure_monthly.csv", parse_dates=["month"])
metro = {"Tempe": "Phoenix", "Scottsdale": "Phoenix", "Chandler": "Phoenix", "Mesa": "Phoenix", "Gilbert": "Phoenix"}
df["metro"] = df["city"].replace(metro)
TOP = ["San Francisco", "Los Angeles", "Austin", "Phoenix", "Atlanta", "Las Vegas"]
W_ORDER = ["Clear", "Cloudy", "Rain", "Other or unknown"]
W_COLORS = ["#d9c89e", "#8f9aa5", "#2f6db3", "#c9c9c9"]
df["weather4"] = df["weather"].where(df["weather"].isin(["Clear", "Cloudy", "Rain"]), "Other or unknown")
SEV_ORDER = ["No injury", "Minor", "Moderate", "Serious", "Fatality", "Unknown"]
SEV_COLORS = ["#dddddd", "#f2b680", "#e8803a", "#c23b22", "#6b0f0f", "#f5f5f5"]

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h

def save(chart, name, w=2):
    """Show the chart in the notebook and also write a PNG copy to charts/ for the report."""
    chart.save(f"charts/{name}.png", scale_factor=w)
    display(chart)

base_cfg = dict(labelFontSize=12, titleFontSize=13)
def finish(ch):
    return ch.configure_axis(**base_cfg).configure_legend(**base_cfg).configure_title(fontSize=16, anchor="start", subtitleFontSize=12, subtitleColor="#555").configure_view(strokeWidth=0)

## 3. Exploration, seven steps

### Step 1. First look: counts by reported weather
Raw counts of all eight weather codes. Rain (114 crashes) is the only adverse condition with enough cases to test; no denominator yet.

In [ ]:
# 1a first look: raw counts by weather (all 8 categories)
c = (alt.Chart(df).mark_bar(color="#4e79a7").encode(
        x=alt.X("count():Q", title="Reported crashes"),
        y=alt.Y("weather:N", sort="-x", title=None))
     .properties(width=520, height=240, title="H1 step 1: crashes by reported weather, all ADS incidents (n = 2,945)"))
save(finish(c), "h1_step1_counts")

### Step 2. Collapse to four groups, shares by metro
Rain is 1 to 6 percent of crashes everywhere and the city order matches how rainy the cities are, which motivates an exposure baseline.

In [ ]:
# 1b: collapse to four groups, normalize, split by metro area
d = df[df["metro"].isin(TOP)]
c = (alt.Chart(d).mark_bar().encode(
        y=alt.Y("metro:N", sort=TOP, title=None),
        x=alt.X("count():Q", stack="normalize", title="Share of crashes", axis=alt.Axis(format="%")),
        color=alt.Color("weather4:N", sort=W_ORDER, scale=alt.Scale(domain=W_ORDER, range=W_COLORS), title="Weather"),
        order=alt.Order("w_idx:Q"),
        tooltip=["metro", "weather4", "count()"])
     .transform_calculate(w_idx=f"indexof({json.dumps(W_ORDER)}, datum.weather4)")
     .properties(width=560, height=230, title={"text": "H1 step 2: weather share of crashes by metro area", "subtitle": "Rain is 1 to 6 percent of crashes everywhere. But how often does it rain in each city?"}))
save(finish(c), "h1_step2_share_by_city")

### Exposure table
Share of crashes in rain versus share of hours with rain per metro, with Wilson intervals on the crash share.

In [ ]:
# exposure table: rain hours share over the months each metro had reports
rows = []
for m in TOP:
    s = d[d["metro"] == m]; months = s["month"].dropna().unique()
    e = ex[(ex["city"] == m) & (ex["month"].isin(months))]
    k = int((s["weather"] == "Rain").sum()); n = len(s)
    p, lo, hi = wilson(k, n); expo = e["rain_hours"].sum() / e["hours"].sum()
    rows.append(dict(metro=m, crashes=n, rain_crashes=k, crash_rain_share=p, lo=lo, hi=hi, rain_hours_share=expo,
                     ratio=p / expo, ratio_lo=lo / expo, ratio_hi=hi / expo))
expo = pd.DataFrame(rows); expo.to_csv("h1_exposure_table.csv", index=False); print(expo.round(3).to_string())

### Step 3. Exposure next to outcome (dumbbell)
In all six metros the crash share sits well below the exposure share. The hypothesis flips here.

In [ ]:
# 1c: dumbbell, crash share vs exposure share
long = expo.melt(id_vars=["metro", "crashes"], value_vars=["crash_rain_share", "rain_hours_share"], var_name="measure", value_name="share")
long["measure"] = long["measure"].map({"crash_rain_share": "Share of crashes in rain", "rain_hours_share": "Share of hours with rain"})
rule = alt.Chart(expo).mark_rule(color="#bbb", strokeWidth=2).encode(y=alt.Y("metro:N", sort=TOP, title=None), x="crash_rain_share:Q", x2="rain_hours_share:Q")
pts = alt.Chart(long).mark_point(size=130, filled=True).encode(
        y=alt.Y("metro:N", sort=TOP), x=alt.X("share:Q", axis=alt.Axis(format="%"), title="Share"),
        color=alt.Color("measure:N", scale=alt.Scale(domain=["Share of crashes in rain", "Share of hours with rain"], range=["#2f6db3", "#9aa5b1"]), title=None),
        tooltip=["metro", "measure", alt.Tooltip("share:Q", format=".1%")])
c = (rule + pts).properties(width=560, height=230, title={"text": "H1 step 3: crashes in rain vs how often it rains, by metro", "subtitle": "Exposure = share of hours with precipitation (Open-Meteo) over the months each city had reports"})
save(finish(c).configure_legend(orient="bottom"), "h1_step3_dumbbell")

### Step 4. Ratio with 95 percent intervals
All ratios between 0.38 and 0.63; four metros exclude 1. Las Vegas rests on one crash.

In [ ]:
# 1d: ratio with 95% interval, reference at 1
expo["ratio_hi_c"] = expo["ratio_hi"].clip(upper=1.55)
c = (alt.Chart(expo).mark_point(size=120, filled=True, color="#2f6db3").encode(
        y=alt.Y("metro:N", sort=alt.EncodingSortField("ratio", order="descending"), title=None),
        x=alt.X("ratio:Q", title="Rain crash share divided by rain hour share (1 = proportional)", scale=alt.Scale(domain=[0, 1.6])),
        tooltip=["metro", "crashes", "rain_crashes", alt.Tooltip("ratio:Q", format=".2f")])
     + alt.Chart(expo).mark_rule(color="#2f6db3", strokeWidth=2).encode(y=alt.Y("metro:N", sort=alt.EncodingSortField("ratio", order="descending")), x="ratio_lo:Q", x2="ratio_hi_c:Q")
     + alt.Chart(pd.DataFrame({"x": [1]})).mark_rule(color="#c23b22", strokeDash=[5, 4]).encode(x="x:Q")
     + alt.Chart(expo).mark_text(align="left", dx=8, dy=1, fontSize=11, color="#555").encode(y=alt.Y("metro:N", sort=alt.EncodingSortField("ratio", order="descending")), x="ratio_hi_c:Q", text=alt.Text("label:N"))
       .transform_calculate(label="datum.rain_crashes + ' of ' + datum.crashes + (datum.ratio_hi > 1.55 ? ' (interval runs to ' + format(datum.ratio_hi, '.1f') + ')' : '')")
    ).properties(width=560, height=240, title={"text": "H1 step 4: in every metro, rain crashes are about half of what rain exposure predicts", "subtitle": "Point = ratio of shares, bar = 95% Wilson interval on the crash share. All six points sit below 1; four intervals exclude it."})
save(finish(c), "h1_step4_ratio_ci")

### Step 5. San Francisco month by month
Rain crashes track the rainy season; total crashes do not. The under representation is not an averaging artifact.

In [ ]:
# 1e: monthly SF: rain crashes vs rain hours
sf = d[d["metro"] == "San Francisco"].groupby("month").agg(crashes=("weather", "size"), rain_crashes=("weather", lambda s: (s == "Rain").sum())).reset_index()
sf = sf.merge(ex[ex["city"] == "San Francisco"][["month", "rain_hours"]], on="month")
sfl = sf.melt(id_vars="month", value_vars=["rain_crashes", "rain_hours"], var_name="series", value_name="value")
sfl["series"] = sfl["series"].map({"rain_crashes": "Crashes reported in rain", "rain_hours": "Hours of rain in the month"})
c = (alt.Chart(sfl).mark_area(opacity=0.85).encode(
        x=alt.X("month:T", title=None), y=alt.Y("value:Q", title=None),
        row=alt.Row("series:N", title=None, header=alt.Header(labelFontSize=12, labelAngle=0, labelAlign="left")),
        color=alt.Color("series:N", scale=alt.Scale(range=["#2f6db3", "#9aa5b1"]), legend=None))
     .resolve_scale(y="independent").properties(width=600, height=110, title={"text": "H1 step 5: San Francisco, month by month", "subtitle": "Rain crashes follow the rainy season (r = 0.56 with rain hours); total crashes do not (r = -0.13)."}))
save(finish(c), "h1_step5_sf_monthly")

### Step 6. Is the risk hiding in severity?
**6a. Stacked counts (dead end).** Rain is a sliver next to clear; the question cannot be read from counts.

In [ ]:
# 2a: severity by weather, stacked counts
d2 = df[df["weather4"] != "Other or unknown"]
c = (alt.Chart(d2).mark_bar().encode(
        x=alt.X("weather4:N", sort=W_ORDER[:3], title=None), y=alt.Y("count():Q", title="Crashes"),
        color=alt.Color("severity:N", sort=SEV_ORDER, scale=alt.Scale(domain=SEV_ORDER, range=SEV_COLORS), title="Highest injury"),
        order=alt.Order("s_idx:Q"))
     .transform_calculate(s_idx=f"indexof({json.dumps(SEV_ORDER)}, datum.severity)")
     .properties(width=360, height=300, title={"text": "H2 step 1: severity by weather, counts", "subtitle": "Rain is a sliver; the stacked counts cannot answer the question."}))
save(finish(c), "h2_step1_counts")

**6b. Normalized stack.** Comparable now, but no sense of uncertainty.

In [ ]:
# 2b: normalized
c = (alt.Chart(d2).mark_bar().encode(
        y=alt.Y("weather4:N", sort=W_ORDER[:3], title=None),
        x=alt.X("count():Q", stack="normalize", axis=alt.Axis(format="%"), title="Share of crashes"),
        color=alt.Color("severity:N", sort=SEV_ORDER, scale=alt.Scale(domain=SEV_ORDER, range=SEV_COLORS), title="Highest injury"),
        order=alt.Order("s_idx:Q"), tooltip=["weather4", "severity", "count()"])
     .transform_calculate(s_idx=f"indexof({json.dumps(SEV_ORDER)}, datum.severity)")
     .properties(width=560, height=160, title={"text": "H2 step 2: severity shares by weather", "subtitle": "Normalized: roughly 90 percent of crashes in every weather are property damage only."}))
save(finish(c), "h2_step2_normalized")

**6c. Injury rates with intervals, weather and surface.** Rain 7.0 percent, clear 10.0, wet 9.7, dry 9.6. No evidence of higher severity in rain.

In [ ]:
# 2c: injury rates with intervals
rows = []
for grp, col in [("weather", "weather4"), ("surface", "surface")]:
    for lvl, s in d2.groupby(col):
        if len(s) < 20: continue
        k = int(s["any_injury"].sum()); n = len(s); p, lo, hi = wilson(k, n)
        rows.append(dict(group=grp, level=lvl, n=n, injured=k, rate=p, lo=lo, hi=hi))
sev = pd.DataFrame(rows); sev.to_csv("h2_injury_rates.csv", index=False); print(sev.round(3).to_string())
sev["label"] = sev["level"] + "  (n = " + sev["n"].astype(str) + ")"
c = (alt.Chart(sev).mark_rule(color="#2f6db3", strokeWidth=2).encode(y=alt.Y("label:N", title=None, sort=alt.EncodingSortField("rate", order="descending")), x="lo:Q", x2="hi:Q")
     + alt.Chart(sev).mark_point(size=140, filled=True, color="#2f6db3").encode(y=alt.Y("label:N", sort=alt.EncodingSortField("rate", order="descending")), x=alt.X("rate:Q", axis=alt.Axis(format="%"), title="Share of crashes with any reported injury", scale=alt.Scale(domain=[0, 0.2])), tooltip=["level", "n", "injured", alt.Tooltip("rate:Q", format=".1%")])
    ).facet(row=alt.Row("group:N", title=None, header=alt.Header(labelFontSize=13, labelAngle=0, labelAlign="left")), spacing=12).resolve_scale(y="independent").properties(title={"text": "H2 step 3: injury rate with 95% intervals", "subtitle": "Rain (7%) and wet roads (10%) are no worse than clear and dry. The rain interval is wide: 8 injuries in 114 crashes."})
save(finish(c), "h2_step3_rates_ci")

**6d. What changes in rain is the crash partner.** More passenger cars and fixed objects, fewer SUVs, heavy trucks and cyclists.

In [ ]:
# 2d: what changes in rain is who you hit, not how badly
pt = d2[d2["weather4"].isin(["Clear", "Rain"])]
ct = pd.crosstab(pt["crash_partner"], pt["weather4"], normalize="columns").reset_index()
ct["diff_pp"] = (ct["Rain"] - ct["Clear"]) * 100
ct = ct[ct[["Clear", "Rain"]].max(axis=1) > 0.02]
c = (alt.Chart(ct).mark_bar().encode(
        y=alt.Y("crash_partner:N", sort=alt.EncodingSortField("diff_pp", order="descending"), title=None),
        x=alt.X("diff_pp:Q", title="Percentage point difference, rain minus clear"),
        color=alt.condition("datum.diff_pp > 0", alt.value("#2f6db3"), alt.value("#9aa5b1")),
        tooltip=["crash_partner", alt.Tooltip("Clear:Q", format=".1%"), alt.Tooltip("Rain:Q", format=".1%")])
     .properties(width=520, height=260, title={"text": "H2 step 4: what the ADS hits in rain, compared with clear weather", "subtitle": "More passenger cars, 'other' objects and fixed objects; fewer SUVs, heavy trucks and cyclists."}))
save(finish(c), "h2_step4_partner_shift")

**6e. Pre crash speed by weather (dead end).** Median 0 mph in every weather group.

In [ ]:
# 2e: pre crash speed distribution by weather (strip + median)
sp = d2.dropna(subset=["speed_mph"])
sp = sp[sp["speed_mph"] <= 60]
c = (alt.Chart(sp).mark_boxplot(extent="min-max", size=28, color="#2f6db3").encode(
        x=alt.X("weather4:N", sort=W_ORDER[:3], title=None), y=alt.Y("speed_mph:Q", title="ADS pre crash speed (mph)"))
     .properties(width=360, height=280, title={"text": "H2 step 5: pre crash speed is the same in rain", "subtitle": "Median speed is 0 mph in all three; most reported crashes involve a stopped or crawling vehicle."}))
save(finish(c), "h2_step5_speed")

### Step 7. Is the risk hiding in one operator or one season?
**7a. Rain share by operator.** Avride and Aurora look rain prone when pooled across cities.

In [ ]:
# 3a: rain share by operator
ops = df.groupby("operator").agg(n=("weather", "size"), rain=("weather", lambda s: (s == "Rain").sum())).reset_index()
ops = ops[ops["n"] >= 20]; ops["rain_share"] = ops["rain"] / ops["n"]
ops["label"] = ops["operator"] + " (n = " + ops["n"].astype(str) + ")"
c = (alt.Chart(ops).mark_bar(color="#2f6db3").encode(
        y=alt.Y("label:N", sort="-x", title=None), x=alt.X("rain_share:Q", axis=alt.Axis(format="%"), title="Share of crashes in rain"),
        tooltip=["operator", "n", "rain"])
     .properties(width=520, height=260, title={"text": "H3 step 1: rain share of crashes by operator", "subtitle": "Avride and Aurora look worst. Is that the operator or where they drive?"}))
save(finish(c), "h3_step1_operator")

**7b. Operators inside each metro, with the city's exposure line.** The operator differences dissolve: every operator sits below its city's exposure line.

In [ ]:
# 3b: same, but faceted by metro, operators with >= 15 crashes in that metro
oc = df[df["metro"].isin(TOP)].groupby(["metro", "operator"]).agg(n=("weather", "size"), rain=("weather", lambda s: (s == "Rain").sum())).reset_index()
oc = oc[oc["n"] >= 15]; oc["rain_share"] = oc["rain"] / oc["n"]
oc = oc.merge(expo[["metro", "rain_hours_share"]], on="metro")
c = (alt.Chart(oc).mark_point(size=120, filled=True).encode(
        x=alt.X("rain_share:Q", axis=alt.Axis(format="%"), title="Share of crashes in rain"),
        y=alt.Y("operator:N", title=None), color=alt.Color("operator:N", legend=None),
        tooltip=["metro", "operator", "n", "rain"])
     + alt.Chart(oc).mark_rule(color="#c23b22", strokeDash=[4, 3]).encode(x="rain_hours_share:Q")
    ).properties(width=200, height=120).facet(facet=alt.Facet("metro:N", sort=TOP, title=None), columns=3).resolve_scale(y="independent").properties(title={"text": "H3 step 2: the same operators inside each metro", "subtitle": "Red dashed line = share of hours with rain in that metro. Every operator sits below its city's exposure line."})
save(finish(c), "h3_step2_operator_by_city")

**7c. Seasonality heatmap.** Rain crashes follow each city's rainy season.

In [ ]:
# 3c: seasonality heatmap, rain crash share by metro and calendar month vs exposure
d3 = df[df["metro"].isin(TOP)].copy(); d3["cal_month"] = d3["month"].dt.month
hm = d3.groupby(["metro", "cal_month"]).agg(n=("weather", "size"), rain=("weather", lambda s: (s == "Rain").sum())).reset_index()
hm["rain_share"] = hm["rain"] / hm["n"]
e2 = ex.copy(); e2["cal_month"] = e2["month"].dt.month
e2 = e2.groupby(["city", "cal_month"]).apply(lambda g: g["rain_hours"].sum() / g["hours"].sum()).rename("expo").reset_index().rename(columns={"city": "metro"})
hm = hm.merge(e2, on=["metro", "cal_month"])
hml = hm.melt(id_vars=["metro", "cal_month", "n"], value_vars=["rain_share", "expo"], var_name="what", value_name="value")
hml["what"] = hml["what"].map({"rain_share": "Share of crashes in rain", "expo": "Share of hours with rain"})
c = (alt.Chart(hml).mark_rect().encode(
        x=alt.X("cal_month:O", title="Calendar month"), y=alt.Y("metro:N", sort=TOP, title=None),
        color=alt.Color("value:Q", scale=alt.Scale(scheme="blues", domain=[0, 0.3]), title="Share", legend=alt.Legend(format="%")),
        tooltip=["metro", "cal_month", "what", alt.Tooltip("value:Q", format=".1%"), "n"])
     .properties(width=480, height=150).facet(row=alt.Row("what:N", title=None, header=alt.Header(labelFontSize=13, labelAngle=0, labelAlign="left")))
     .properties(title={"text": "H3 step 3: rain crashes follow each city's rainy season", "subtitle": "Top: when ADS crashes happen in rain. Bottom: when it rains. Winter in California, summer in Texas and Georgia."}))
save(finish(c), "h3_step3_season_heatmap")

**7d. Monthly small multiples.** Crash share in rain tracks exposure where data is dense (SF and LA, r = 0.69).

In [ ]:
# 3d: monthly rain crashes vs rain hours, small multiples per metro (months with at least 5 crashes)
mm = d3.groupby(["metro", "month"]).agg(rain=("weather", lambda s: (s == "Rain").sum()), n=("weather", "size")).reset_index()
mm = mm.merge(ex.rename(columns={"city": "metro"})[["metro", "month", "rain_hours", "hours"]], on=["metro", "month"])
mm = mm[mm["n"] >= 5]
mm["rain_crash_share"] = mm["rain"] / mm["n"]
mm["exposure"] = mm["rain_hours"] / mm["hours"]
corr = mm.groupby("metro").apply(lambda g: g["rain_crash_share"].corr(g["exposure"])).round(2).to_dict(); print("monthly corr", corr)
mml = mm.melt(id_vars=["metro", "month", "n"], value_vars=["rain_crash_share", "exposure"], var_name="series", value_name="value")
mml["series"] = mml["series"].map({"rain_crash_share": "Share of crashes in rain", "exposure": "Share of hours with rain"})
panels = []
for m in TOP:
    sub = mml[mml["metro"] == m]
    panels.append(alt.Chart(sub).mark_line(clip=True).encode(
        x=alt.X("month:T", title=None, scale=alt.Scale(domain=[pd.Timestamp("2021-07-01"), pd.Timestamp("2026-08-01")])),
        y=alt.Y("value:Q", axis=alt.Axis(format="%"), title=None, scale=alt.Scale(domain=[0, 0.4])),
        color=alt.Color("series:N", scale=alt.Scale(range=["#2f6db3", "#9aa5b1"]), title=None),
        strokeWidth=alt.condition("datum.series == 'Share of crashes in rain'", alt.value(2.2), alt.value(1.2)),
        tooltip=["metro", "month:T", "series", alt.Tooltip("value:Q", format=".0%"), "n"])
      .properties(width=260, height=120, title=alt.TitleParams(f"{m}  (r = {corr.get(m, float('nan')):.2f})", fontSize=12, anchor="start")))
c = alt.concat(*panels, columns=2, spacing=14).properties(title={"text": "H3 step 4: month by month, rain crash share tracks rain exposure where the data is dense (SF, LA: r = 0.69)", "subtitle": "Blue: share of that month's crashes reported in rain. Gray: share of hours with rain. Months with fewer than 5 crashes dropped. The four newer metros are too sparse to judge yet."})
save(finish(c).configure_legend(orient="bottom"), "h3_step4_monthly_small_multiples")

**7e. Lighting by weather (2021 to 2025 reports only).** About 40 percent of crashes are in the dark in every weather; rain at night is not special.

In [ ]:
# 3e: lighting x weather (archive only) - does rain at night look different
la = df.dropna(subset=["lighting"]); la = la[la["weather4"].isin(["Clear", "Cloudy", "Rain"]) & la["lighting"].isin(["Daylight", "Dark - Lighted", "Dawn / Dusk", "Dark - Not Lighted"])]
c = (alt.Chart(la).mark_bar().encode(
        y=alt.Y("weather4:N", sort=W_ORDER[:3], title=None),
        x=alt.X("count():Q", stack="normalize", axis=alt.Axis(format="%"), title="Share of crashes"),
        color=alt.Color("lighting:N", sort=["Daylight", "Dawn / Dusk", "Dark - Lighted", "Dark - Not Lighted"], scale=alt.Scale(range=["#f3d98b", "#e8a44a", "#5b6c84", "#1f2a3a"]), title="Lighting"),
        tooltip=["weather4", "lighting", "count()"])
     .properties(width=560, height=150, title={"text": "H3 step 5: lighting at the time of the crash, by weather (2021 to mid 2025 reports only)", "subtitle": "About 40 percent of crashes are in the dark whatever the weather; rain at night is not over represented."}))
save(finish(c), "h3_step5_lighting")

## 4. Note for the team: pre crash speed

A quick look at the suggested speed field. It is recorded for 99.7 percent of crashes; in 54 percent the ADS was at 0 mph (stopped or parked and struck by someone else). Among moving vehicles the injury rate barely rises with ADS speed. That supports a separate hypothesis: most ADS crashes are not caused by the ADS vehicle's motion.

In [ ]:
s = df.dropna(subset=["speed_mph"]); s = s[s["speed_mph"] <= 80]
s["bin"] = pd.cut(s["speed_mph"], [-1, 0, 5, 15, 25, 35, 80], labels=["0", "1-5", "6-15", "16-25", "26-35", "36+"])
g = s.groupby("bin", observed=True).agg(n=("any_injury", "size"), injury_rate=("any_injury", "mean")).reset_index()
print("share of crashes with ADS at 0 mph:", round((df["speed_mph"] == 0).mean(), 3))
alt.Chart(g).mark_bar(color="#2f6db3").encode(x=alt.X("bin:N", title="ADS pre crash speed (mph)", sort=None), y=alt.Y("injury_rate:Q", axis=alt.Axis(format="%"), title="Share with any injury"), tooltip=["bin", "n", alt.Tooltip("injury_rate:Q", format=".1%")]).properties(width=420, height=220, title="Injury rate by ADS pre crash speed")

## 7. Midterm slide charts

The four charts on the Hypothesis 3 slides of the midterm deck, drawn from the same cleaned table and exposure file as the sections above. Saved to `charts/slide_*.png`.


In [ ]:
# ---- Midterm slide charts (Hypothesis 3). Same data as above; one PNG per slide in charts/slide_*.png
BLUE, INK, LIGHT, GRAY, RED = "#2E75B6", "#1F3864", "#D9DEE5", "#8A949E", "#C0392B"
DECK = {"config": {"view": {"stroke": None}, "axis": {"labelColor": "#4A5563", "titleColor": "#4A5563", "gridColor": "#ECEFF3", "domainColor": "#CCD3DB", "tickColor": "#CCD3DB", "labelFontSize": 11, "titleFontSize": 11},
        "title": {"color": INK, "fontSize": 15, "anchor": "start", "subtitleColor": "#6B7682", "subtitleFontSize": 11.5}, "legend": {"labelFontSize": 11, "titleFontSize": 11, "labelLimit": 600}}}
alt.theme.register("deck", enable=True)(lambda: DECK)
def slide(ch, name):
    ch.save(f"charts/slide_{name}.png", scale_factor=3); display(ch)

# Slide: weather counts
wc = df["weather"].value_counts().rename_axis("weather").reset_index(name="n"); tot = len(df)
wc["label"] = wc.apply(lambda r: f"{r.n:,} ({r.n / tot:.1%})", axis=1)
b = alt.Chart(wc).encode(y=alt.Y("weather:N", sort="-x", title=None))
slide((b.mark_bar(cornerRadiusEnd=2).encode(x=alt.X("n:Q", title="ADS crash reports, Jul 2021 to Aug 2026", scale=alt.Scale(domain=[0, 2700])),
            color=alt.condition(alt.datum.weather == "Rain", alt.value(BLUE), alt.value(LIGHT)))
       + b.mark_text(align="left", dx=5, color="#4A5563").encode(x="n:Q", text="label:N"))
      .properties(width=560, height=250, title={"text": f"Reported weather at the time of the crash, {tot:,} ADS incidents",
            "subtitle": "Rain is the only adverse condition with enough cases to test; snow, fog and wind together are six crashes."}), "weather_counts")

# Slide: predicted vs reported rain crashes by metro
rows = []
for m in TOP:
    s = df[df["metro"] == m]; months = s["month"].dropna().unique(); e = ex[(ex["city"] == m) & (ex["month"].isin(months))]
    expo = e["rain_hours"].sum() / e["hours"].sum(); k = int((s["weather"] == "Rain").sum()); n = len(s)
    rows.append(dict(metro=m, crashes=n, rainHours=expo, reported=k, predicted=n * expo, ratio=k / (n * expo)))
ext = pd.DataFrame(rows)
KINDS = ["Predicted if rain were neutral (crashes × share of hours with rain)", "Reported in rain"]
panels = []
for i, r in ext.iterrows():
    sub = pd.DataFrame({"kind": KINDS, "value": [r.predicted, r.reported]}); sub["lbl"] = sub["value"].round(0).astype(int)
    bb = alt.Chart(sub).mark_bar(size=34).encode(x=alt.X("kind:N", axis=None, title=None, sort=KINDS),
        y=alt.Y("value:Q", title="Rain crashes" if i == 0 else None, scale=alt.Scale(domain=[0, 112]), axis=alt.Axis(labels=i == 0, ticks=i == 0)),
        color=alt.Color("kind:N", title=None, scale=alt.Scale(domain=KINDS, range=[LIGHT, BLUE]), legend=alt.Legend(orient="top", direction="vertical", columns=1)))
    tt = alt.Chart(sub).mark_text(dy=-7, fontSize=11).encode(x=alt.X("kind:N", sort=KINDS), y="value:Q", text="lbl:Q", color=alt.condition(alt.datum.kind == "Reported in rain", alt.value(INK), alt.value("#6B7682")))
    rt = alt.Chart(pd.DataFrame({"y": [max(r.predicted, r.reported) + 11], "t": [f"{r.ratio:.2f}×"]})).mark_text(fontSize=12, fontWeight="bold", color=RED).encode(x=alt.value(40), y="y:Q", text="t:N")
    panels.append((bb + tt + rt).properties(width=80, height=230, title=alt.TitleParams(text=r.metro, subtitle=[f"{int(r.crashes):,} crashes", f"rain {r.rainHours:.0%} of hours"], fontSize=12, subtitleFontSize=10, anchor="middle", orient="bottom", color=INK)))
slide(alt.hconcat(*panels, spacing=6).properties(title={"text": "Rain crashes by metro: predicted from rain hours vs reported",
      "subtitle": f"Six metros: {ext.predicted.sum():.0f} predicted if rain were neutral, {int(ext.reported.sum())} reported. Red = reported ÷ predicted, below 1 in every metro."}), "predicted_vs_reported")

# Slide: San Francisco month by month, three stacked panels (no dual axis)
sfm = df[df["metro"] == "San Francisco"].groupby("month").agg(crashes=("weather", "size"), rain=("weather", lambda s: int((s == "Rain").sum()))).reset_index()
sfm = sfm.merge(ex[ex["city"] == "San Francisco"][["month", "rain_hours"]], on="month", how="inner")
r_rain = np.corrcoef(sfm["rain"], sfm["rain_hours"])[0, 1]; r_all = np.corrcoef(sfm["crashes"], sfm["rain_hours"])[0, 1]
xq = alt.X("yearmonth(month):T", title=None, axis=alt.Axis(labels=False, ticks=False, tickCount="year"))
xa = alt.X("yearmonth(month):T", title=None, axis=alt.Axis(format="%Y", tickCount="year", labelAngle=0))
p1 = alt.Chart(sfm).mark_bar(color=LIGHT, width=6).encode(x=xq, y=alt.Y("rain_hours:Q", title="Rain hours")).properties(width=600, height=110)
p2 = alt.Chart(sfm).mark_bar(color=BLUE, width=6).encode(x=xq, y=alt.Y("rain:Q", title="Rain crashes")).properties(width=600, height=110)
p3 = alt.Chart(sfm).mark_line(color=GRAY, strokeWidth=2).encode(x=xa, y=alt.Y("crashes:Q", title="All crashes")).properties(width=600, height=110)
note = lambda t, c, bold: alt.Chart(pd.DataFrame({"t": [t]})).mark_text(align="left", fontSize=11.5, color=c, fontWeight="bold" if bold else "normal").encode(x=alt.value(6), y=alt.value(12), text="t:N")
slide(alt.vconcat(p1, p2 + note(f"r = {r_rain:.2f} with rain hours", BLUE, True), p3 + note(f"r = {r_all:.2f} with rain hours: totals track the size of the fleet, not the weather".replace("-", "−"), "#6B7682", False), spacing=10)
      .properties(title={"text": "San Francisco month by month: rain crashes follow the rainy season, all crashes do not",
                         "subtitle": f"{len(sfm)} months. Hours of rain from the Open-Meteo archive; crashes from the NHTSA reports."}), "sf_monthly")

# Slide: injury rates with Wilson intervals
inj = []
for lvl, s in df[df["weather"].isin(["Clear", "Cloudy", "Rain"])].groupby("weather"):
    p, lo, hi = wilson(int(s["any_injury"].sum()), len(s)); inj.append(dict(group="By weather", level=lvl, n=len(s), rate=p, lo=lo, hi=hi))
for lvl, s in df[df["surface"].isin(["Dry", "Wet"])].groupby("surface"):
    p, lo, hi = wilson(int(s["any_injury"].sum()), len(s)); inj.append(dict(group="By surface", level=lvl, n=len(s), rate=p, lo=lo, hi=hi))
inj = pd.DataFrame(inj); inj["hl"] = inj["level"].isin(["Rain", "Wet"]); inj["rate_lbl"] = inj["rate"].map(lambda v: f"{v:.1%}"); inj["n_lbl"] = inj["n"].map(lambda v: f"n = {v:,}")
overall = df["any_injury"].mean()
def inj_panel(grp, levels, first):
    sub = inj[inj["group"] == grp]; y = alt.Y("level:N", title=None, sort=levels); col = alt.condition(alt.datum.hl, alt.value(BLUE), alt.value(GRAY))
    xs = alt.X("rate:Q", title=None if first else "Share of crashes with any reported injury, 95% Wilson interval", axis=alt.Axis(format=".0%", labels=not first, ticks=not first), scale=alt.Scale(domain=[0, 0.18]))
    layers = [alt.Chart(sub).mark_rule(strokeWidth=5, opacity=0.45, strokeCap="round").encode(x=alt.X("lo:Q", scale=alt.Scale(domain=[0, 0.18])), x2="hi:Q", y=y, color=col),
              alt.Chart(sub).mark_circle(size=140, opacity=1).encode(x=xs, y=y, color=col),
              alt.Chart(sub).mark_text(dy=-13, fontSize=11.5, fontWeight="bold", color=INK).encode(x="rate:Q", y=y, text="rate_lbl:N"),
              alt.Chart(sub).mark_text(align="left", dx=8, fontSize=10.5, color="#8591A0").encode(x="hi:Q", y=y, text="n_lbl:N"),
              alt.Chart(sub).mark_rule(strokeDash=[4, 4], color="#8591A0").encode(x=alt.datum(overall))]
    if not first:
        layers.append(alt.Chart(pd.DataFrame({"t": [f"all crashes {overall:.1%}"]})).mark_text(align="left", dx=4, fontSize=10.5, color="#8591A0").encode(x=alt.datum(overall), y=alt.value(30 * len(levels) + 12), text="t:N"))
    return alt.layer(*layers).properties(width=560, height=30 * len(levels) + 20, title=alt.TitleParams(grp.upper(), fontSize=10.5, color=BLUE, anchor="start", dy=-2))
slide(alt.vconcat(inj_panel("By weather", ["Clear", "Cloudy", "Rain"], True), inj_panel("By surface", ["Dry", "Wet"], False), spacing=16)
      .properties(title={"text": "Rain crashes are no more likely to injure anyone", "subtitle": "Clear 10.0%, cloudy 8.4%, rain 7.0%; dry 9.6%, wet 9.6%. Nothing points up in rain."}), "injury_rates")
